In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

# check if workding_dir is in local variables
if "workding_dir" not in locals():
    workding_dir = str(Path.cwd().parent)

os.chdir(workding_dir)
sys.path.append(workding_dir)
print("workding dir:", workding_dir)

workding dir: /Users/inflaton/code/engd/papers/multi-agent-llm-at-edge


In [3]:
from dotenv import find_dotenv, load_dotenv

found_dotenv = find_dotenv(".env")

if len(found_dotenv) == 0:
    found_dotenv = find_dotenv(".env.example")
print(f"loading env vars from: {found_dotenv}")
load_dotenv(found_dotenv, override=True)

loading env vars from: /Users/inflaton/code/engd/papers/multi-agent-llm-at-edge/.env


True

In [4]:
from src.misc.metrics import *

/Users/inflaton/anaconda3/envs/multi-agent-llm-at-edge/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import pandas as pd

df_emails = pd.read_csv("dataset/emails.csv")
df_emails.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   email_id             1980 non-null   object 
 1   sender_email         1980 non-null   object 
 2   recipient_email      1980 non-null   object 
 3   subject              1980 non-null   object 
 4   email_body           1980 non-null   object 
 5   attachments          990 non-null    object 
 6   process_status       1980 non-null   object 
 7   response             0 non-null      float64
 8   start_time           0 non-null      float64
 9   end_time             0 non-null      float64
 10  full_logs            0 non-null      float64
 11  total_time           1980 non-null   int64  
 12  successful_requests  1980 non-null   int64  
 13  total_tokens         1980 non-null   int64  
 14  prompt_tokens        1980 non-null   int64  
 15  completion_tokens    1980 non-null   i

In [6]:
df_transactions = pd.read_csv("dataset/transactions.csv")
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   invoice_id            1980 non-null   int64  
 1   bank_name             1980 non-null   object 
 2   transaction_id        1980 non-null   object 
 3   amount                1980 non-null   float64
 4   recipient_name        1980 non-null   object 
 5   sender_name           1980 non-null   object 
 6   reconciliation_state  1980 non-null   object 
 7   email_details         0 non-null      float64
dtypes: float64(2), int64(1), object(5)
memory usage: 123.9+ KB


In [7]:
df_ground_truth = pd.read_csv("dataset/ground_truth.csv")
df_ground_truth.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   invoice_id           1980 non-null   int64  
 1   bank_name            1980 non-null   object 
 2   transaction_id       1980 non-null   object 
 3   transaction_date     1980 non-null   object 
 4   amount               1980 non-null   float64
 5   recipient_name       1980 non-null   object 
 6   sender_name          1980 non-null   object 
 7   email_id             1980 non-null   object 
 8   sender_email         1980 non-null   object 
 9   recipient_email      1980 non-null   object 
 10  subject              1980 non-null   object 
 11  email_body           1980 non-null   object 
 12  filename             990 non-null    object 
 13  expected_ocr_result  990 non-null    object 
dtypes: float64(1), int64(1), object(12)
memory usage: 216.7+ KB


In [8]:
df_ground_truth.iloc[0]

invoice_id                                                         43925
bank_name                                                       Bank 538
transaction_id                                          EC4F9FE854344D25
transaction_date                                                 4/23/23
amount                                                             196.0
recipient_name                                                     Tanya
sender_name                                                 Robin Levine
email_id                                93185A89130149C0A842968E4AFDCAA2
sender_email                                     RobinLevine@example.com
recipient_email                             tanya.official.456@gmail.com
subject                       Payment Confirmation for Invoice ID: 43925
email_body             Hi Tanya ! Please find attached payment screen...
filename                                         imgs/transaction_1.jpeg
expected_ocr_result    Bank 538\nSent by: Robin Lev

In [10]:
def get_ocr_result(row):
    return (
        f"""{row['bank_name']}
Sent by: {row['sender_name']}
You paid ${row['amount']:.0f} to {row['recipient_name']}
Transaction Reference Number: {row['transaction_id']}
Transaction Date: {row['transaction_date']}
Invoice ID: {row['invoice_id']}
"""
        if isinstance(row["filename"], str)
        else None
    )


df_ground_truth["expected_ocr_result"] = df_ground_truth.apply(get_ocr_result, axis=1)
df_ground_truth.iloc[0]

invoice_id                                                         43925
bank_name                                                       Bank 538
transaction_id                                          EC4F9FE854344D25
transaction_date                                                 4/23/23
amount                                                             196.0
recipient_name                                                     Tanya
sender_name                                                 Robin Levine
email_id                                93185A89130149C0A842968E4AFDCAA2
sender_email                                     RobinLevine@example.com
recipient_email                             tanya.official.456@gmail.com
subject                       Payment Confirmation for Invoice ID: 43925
email_body             Hi Tanya ! Please find attached payment screen...
filename                                         imgs/transaction_1.jpeg
expected_ocr_result    Bank 538\nSent by: Robin Lev

In [11]:
df_ground_truth.iloc[-1]

invoice_id                                                       1042392
bank_name                                                       Bank 590
transaction_id                                        T-418BBE70721C46FD
transaction_date                                                12/20/24
amount                                                             147.0
recipient_name                                                     Tanya
sender_name                                                    Ruth West
email_id                              E-246AB042D6C74C53B38B68BDBDA07590
sender_email                                        RuthWest@example.com
recipient_email                             tanya.official.456@gmail.com
subject                     Payment Confirmation for Invoice ID: 1042392
email_body             Dear Tanya, I hope this message finds you well...
filename                                                             NaN
expected_ocr_result                                

In [12]:
df_ground_truth.to_csv("dataset/ground_truth.csv", index=False)